# 🌕 ICARUS_WORKER_PLANETARY — Planetary Science Worker
## Dedicated Remote Scientific Worker Runtime (Kaggle Account 3/4)

**Role:** Planetary Surface Science, Lunar Morphology & Topography  
**Primary Model:** NASA-IBM Lunar Foundation Model (`nasa-ibm-ai4science/NASA-IBM-Lunar-Foundation-Model`)  
**Architecture:** Vision Transformer (ViT-B) trained on LROC NAC/WAC and LOLA DEMs  
**Data Sources:** NASA LROC (Narrow/Wide Angle Cameras), LOLA Gridded Data Records (GDR)  
**Deployment Profile:** Kaggle GPU (T4 or P100)

---

### Worker Responsibilities
- Hosts deterministic planetary algorithms: **Horn gradient slope & aspect, terrain roughness index, crater circularity & depth-to-diameter ratio, crater scaling laws**.
- Hosts **NASA-IBM Lunar Foundation Model** for lunar surface representation token extraction.
- Serves `/health`, `/capabilities`, `/analyze`, and `/unload-model` endpoints to `ICARUS_CORE`.


In [ ]:
# 1. Environment & Dependencies Setup
import os
import sys
from pathlib import Path

!pip install -q huggingface_hub rasterio pyngrok uvicorn fastapi pydantic scipy timm

!git clone https://github.com/kanishk10-ux/ICARUS.git /kaggle/working/ICARUS 2>/dev/null || (cd /kaggle/working/ICARUS && git pull)
sys.path.insert(0, "/kaggle/working/ICARUS")

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Lunar Foundation Adapter Verification & Smoke Test
import numpy as np
from icarus.workers.planetary.lunar_adapter import LunarFoundationAdapter
from icarus.algorithms.planetary import PlanetaryAlgorithms

device = "cuda" if torch.cuda.is_available() else "cpu"
adapter = LunarFoundationAdapter(device=device)

ok, msg = adapter.load_model()
print("Load Status:", ok, "->", msg)

# Synthetic Lunar DEM patch: 224x224 elevation grid (meters)
x = np.linspace(-1000, 1000, 224)
y = np.linspace(-1000, 1000, 224)
xx, yy = np.meshgrid(x, y)
crater_dem = 500.0 - 200.0 * np.exp(-(xx**2 + yy**2) / (2 * 200.0**2))

res = adapter.extract_features(crater_dem, observation_id="LROC_NAC_SYNTH_001")
print("Lunar FM Output Status:", res["status"])
print(f"Feature Shape: {res['feature_shape']} ({res['total_patch_tokens']} tokens x {res['hidden_dimension']} dims)")

# Algorithmic Planetary Topography & Morphometry
topo = PlanetaryAlgorithms.calculate_dem_slope_aspect(crater_dem, pixel_scale_meters=10.0)
morph = PlanetaryAlgorithms.compute_crater_morphometry(diameter_m=400.0, depth_m=180.0, perimeter_m=1280.0, area_m2=125663.0)
print(f"Mean Surface Slope: {topo['mean_slope_degrees']:.2f} deg, Max Slope: {topo['max_slope_degrees']:.2f} deg")
print(f"Crater Classification: {morph['morphology_class']} (d/D: {morph['depth_to_diameter_ratio']:.3f}, circularity: {morph['circularity_index']:.3f})")

In [ ]:
# 3. Launch Dedicated Worker Service with Public ngrok Tunnel
from scripts.worker_bootstrap import create_worker_app
import threading
import uvicorn

app = create_worker_app("planetary", device=device)

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8003, log_level="warning")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
print("✅ Planetary Worker running locally on port 8003")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8003, "http")
    public_url = tunnel.public_url
    print(f"🚀 PUBLIC NGROK ENDPOINT: {public_url}")
    print("\nUse this URL to register the worker with ICARUS_CORE:")
    print(f"Worker Registration Command for ICARUS_CORE:")
    print(f'curl -X POST <ICARUS_CORE_URL>/workers/register -H "Content-Type: application/json" -d \'{"worker_id": "worker-planetary-01", "domain": "Planetary Science", "endpoint": "{public_url}", "status": "READY", "capabilities": ["dem_slope_aspect", "terrain_roughness", "crater_morphometry", "crater_scaling_law", "lunar_representation_tokens"]}\''')
except Exception as e:
    print("ngrok tunnel configuration note (Kaggle Secret NGROK_AUTH_TOKEN required for public access):", e)

In [ ]:
# 4. Worker Self-Test HTTP Loopback Verification
import requests

base_url = "http://127.0.0.1:8003"
health = requests.get(f"{base_url}/health").json()
print("Worker Health Status:", health)

caps = requests.get(f"{base_url}/capabilities").json()
print(f"Registered Capabilities: {len(caps['capabilities'])} operations supported")

# Test Crater Morphometry Analysis via API
analysis_req = {
    "task_id": "test-crater-morph-01",
    "operation": "crater_morphometry",
    "parameters": {
        "diameter_m": 500.0,
        "depth_m": 120.0,
        "perimeter_m": 1570.0,
        "area_m2": 196350.0
    }
}
analysis_resp = requests.post(f"{base_url}/analyze", json=analysis_req).json()
print("Crater Analysis Response:", analysis_resp)